
# Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.



If your bucket doesn't already exist: Run the following cell to create your Cloud Storage bucket.

In [ ]:
#! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

# Task - 1

Activate GCP & Setup Vertex AI Workbench

Activate your GCP trial account and set up a Vertex AI Workbench instance. Enable all appropriate services and APIs as required.

solution : Yes, Its now done! Now were are in the jupyter notebook of the workbech.

# TASK 2

Store Training Data in GCS

Create a Google Cloud Storage bucket and upload the IRIS dataset to it. Split the data into training and evaluation sets as you deem fit.

In [ ]:
from google.cloud import storage

import pandas as pd

from sklearn.model_selection import train_test_split
from io import StringIO

In [ ]:
PROJECT_ID = "iitm-mlops-assignments "  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
BUCKET_URI = f"gs://iitm-mlops-assignments"  # @param {type:"string"}
MODEL_ARTIFACT_DIR="week-1"
BUCKET_NAME = "iitm-mlops-assignments"
DATASET_RAW="data/raw"
DATASET_V1="data/v1"
DATASET_V2="data/v2"

In [ ]:
#Uploading the dataset into the Cloud Storage once executed don't reexecute it. 

client = storage.Client(project=PROJECT_ID)

bucket = client.bucket(BUCKET_NAME)

#Uploading the raw dataset in GCP
blob = bucket.blob("week-1/data/raw/iris.csv")

blob.upload_from_filename("data/iris.csv")

#Uploading the v1 dataset in GCP
blob = bucket.blob("week-1/data/v1/data.csv")

blob.upload_from_filename("data/data_v1.csv")

#Uploading the v2 dataset in GCP
blob = bucket.blob("week-1/data/v2/data.csv")

blob.upload_from_filename("data/data_v2.csv")

In [ ]:
#revoking the cloud date for training the model.

blob = bucket.blob("week-1/data/raw/iris.csv")

csv_data = blob.download_as_text()

df = pd.read_csv(StringIO(csv_data))

df.head()

In [ ]:
print(df.shape)

df.head()

df.info()

In [ ]:
#spliting the data into train and test

X = df.drop(columns=['species'])

y  = df['species']

x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.2,  random_state=42, stratify=y)

print("Training Samples",len(x_train))

print("Validation Samples", len(x_val))

# Task 3

Execute the IRIS Training Pipeline

Fetch the data from the GCS bucket and successfully execute the IRIS machine learning training pipeline. Store output artifacts (models, logs, etc.) in a GCS bucket with folders organized by their training execution timestamp.

In [ ]:
from google.cloud import storage

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

import joblib
import pickle

from datetime import datetime
from io import StringIO
import json

In [ ]:
blob = bucket.blob("week-1/data/raw/iris.csv")

csv_data = blob.download_as_text()

df = pd.read_csv(StringIO(csv_data))

In [ ]:
X = df.drop(columns=['species'])
y = df['species']

In [ ]:
x_train, x_val, y_train,y_val = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(x_train,y_train)

In [ ]:
#Evaluating the model
accuracy = accuracy_score(y_val,model.predict(x_val))

print("The model accuracy in the raw validation set is to be ", accuracy)

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Run ID:", timestamp)

In [ ]:
# model
joblib.dump(model, "artifacts/model.joblib")

# metrics
metrics = {"accuracy": float(accuracy)}
with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f)

# predictions
pred_df = x_val.copy()
pred_df["actual"] = y_val.values
pred_df["predicted"] = model.predict(x_val)
pred_df.to_csv("artifacts/predictions.csv", index=False)

In [ ]:
files = ["model.joblib", "metrics.json", "predictions.csv"]

for file in files:
    blob = bucket.blob(f"week-1/artifacts/{timestamp}/{file}")
    blob.upload_from_filename(f"artifacts/{file}")

print("Artifacts uploaded successfully!")

# Task 4

Run Inference on Evaluation Set

Create a separate inference script that fetches the trained model from the GCS output artifacts bucket and runs inference on the evaluation set.

In [ ]:
import joblib
from google.cloud import storage
import pandas as pd
from io import StringIO

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

model_blob = bucket.blob(f"week-1/artifacts/{timestamp}/model.joblib")
model_blob.download_to_filename("model.joblib")

model = joblib.load("model.joblib")

In [ ]:
#loading evaluation data

blob = bucket.blob("week-1/data/raw/iris.csv")
csv_data = blob.download_as_text()

df = pd.read_csv(StringIO(csv_data))

X = df.drop(columns=["species"])
y= df["species"]

x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

In [ ]:
predictions = model.predict(x_val)

In [ ]:
print("The Inference accuracy score on validation data of the model loaded in cloud storage", accuracy_score(y_val,predictions))

# Task -5

Run Training & Inference Twice

Execute the training and inference pipeline two times, resulting in two separate output artifact folders in your GCS bucket (each organized by execution timestamp).

Solution : Yes I had ran the model and now I have two different verisons of the artificats organized by their timestamps

# Task - 6

Multiple Data Versions

Run the pipeline for two different versions of data provided in the GitHub data folder. Compare the results across data versions.

In [ ]:
#Importing the required librararies 
from google.cloud import storage

import pandas as pd
import os

from io import StringIO
from datetime import datetime
import joblib
import json

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
client = storage.Client(project=PROJECT_ID)
blob = client.bucket(BUCKET_NAME)

v1 = bucket.blob("week-1/data/v1/data.csv")
v2 = bucket.blob("week-1/data/v2/data.csv")

csv_v1 = v1.download_as_text()
csv_v2 = v2.download_as_text()

df_v1 = pd.read_csv(StringIO(csv_v1))
df_v2 = pd.read_csv(StringIO(csv_v2))

In [ ]:
def run_pipeline(data, bucket,version):
    X = data.drop(columns=["species"])
    y = data["species"]
    x_train, x_val, y_train, y_val = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
    
    model = DecisionTreeClassifier(max_depth=3, random_state=42)
    model.fit(x_train, y_train)

    score = accuracy_score(y_val,model.predict(x_val))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    os.makedirs("artifacts",exist_ok=True)

    joblib.dump(model,f"artifacts/{version}/model.joblib")

    metrics = {
        "version" : version,
        "accuracy" : float(score)
    }

    with open(f"artifacts/{version}/metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)
    
    pred_df = x_val.copy()
    pred_df["actual"] = y_val.values
    pred_df["predicted"] = model.predict(x_val)
    pred_df.to_csv(f"artifacts/{version}/predictions.csv", index=False)

    files = ["model.joblib","metrics.json","predictions.csv"]
    for file in files:
        blob = bucket.blob(f"week-1/artifacts/{version}/{timestamp}/{file}")
        blob.upload_from_filename(f"artifacts/{version}/{file}")
    
    print(f"Pipeline completed for {version} | Accuracy: {score}")    

In [ ]:
run_pipeline(df_v1, bucket, "v1")
run_pipeline(df_v2, bucket, "v2")

In [ ]:
# Other way to upload files to GCP. Not used currently. One can also use similar command to get files from cloud storage using this command. 
#!gsutil cp artifacts/model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/